In [2]:
import pandas as pd

RAW = ""

orders       = pd.read_csv(RAW + "olist_orders_dataset.csv")
customers    = pd.read_csv(RAW + "olist_customers_dataset.csv")
order_items  = pd.read_csv(RAW + "olist_order_items_dataset.csv")
payments     = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews      = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
products     = pd.read_csv(RAW + "olist_products_dataset.csv")
sellers      = pd.read_csv(RAW + "olist_sellers_dataset.csv")
geolocation  = pd.read_csv(RAW + "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW + "product_category_name_translation.csv")

print("Loaded all 9 tables")

Loaded all 9 tables


In [3]:
tables = {
    "orders": orders, "customers": customers, "order_items": order_items,
    "payments": payments, "reviews": reviews, "products": products,
    "sellers": sellers, "geolocation": geolocation,
    "category_translation": category_translation,
}

for name, df in tables.items():
    print(f"{name:22} rows={df.shape[0]:>8}  cols={df.shape[1]}")

orders                 rows=   99441  cols=8
customers              rows=   99441  cols=5
order_items            rows=  112650  cols=7
payments               rows=  103886  cols=5
reviews                rows=   99224  cols=7
products               rows=   32951  cols=9
sellers                rows=    3095  cols=4
geolocation            rows= 1000163  cols=5
category_translation   rows=      71  cols=2


In [4]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print("duplicate rows:", df.duplicated().sum())
    nulls = df.isnull().sum()
    print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "no nulls")


=== orders ===
duplicate rows: 0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

=== customers ===
duplicate rows: 0
no nulls

=== order_items ===
duplicate rows: 0
no nulls

=== payments ===
duplicate rows: 0
no nulls

=== reviews ===
duplicate rows: 0
review_comment_title      87656
review_comment_message    58247

=== products ===
duplicate rows: 0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

=== sellers ===
duplicate rows: 0
no nulls

=== geolocation ===
duplicate rows: 261831
no nulls

=== category_translation ===
duplicate rows: 0
no nulls


In [5]:
for name in ["products", "sellers", "geolocation"]:
    df = tables[name]
    print(f"\n=== {name} ===")
    print("duplicate rows:", df.duplicated().sum())
    nulls = df.isnull().sum()
    print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "no nulls")


=== products ===
duplicate rows: 0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

=== sellers ===
duplicate rows: 0
no nulls

=== geolocation ===
duplicate rows: 261831
no nulls


In [6]:
# 1. Convert date columns from text to real dates
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

# 2. New column: days taken to deliver
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

# 3. English category names; fill missing with "unknown"
products = products.merge(category_translation, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")

# 4. One payment row per order
pay = payments.groupby("order_id").agg(
    total_payment=("payment_value", "sum"),
    payment_type=("payment_type", "first")
).reset_index()

# 5. One review row per order (keep the latest)
rev = (reviews.sort_values("review_creation_date")
              .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])

# 6. Join everything into one table
fact = (order_items
        .merge(orders, on="order_id", how="left")
        .merge(customers, on="customer_id", how="left")
        .merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")
        .merge(sellers, on="seller_id", how="left")
        .merge(pay, on="order_id", how="left")
        .merge(rev, on="order_id", how="left"))

print(fact.shape)
fact.head()

(112650, 26)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,customer_zip_code_prefix,customer_city,customer_state,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,total_payment,payment_type,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,28013,campos dos goytacazes,RJ,cool_stuff,27277,volta redonda,SP,72.19,credit_card,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,15775,santa fe do sul,SP,pet_shop,3471,sao paulo,SP,259.83,credit_card,4.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,35661,para de minas,MG,furniture_decor,37564,borda da mata,MG,216.87,credit_card,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,12952,atibaia,SP,perfumery,14403,franca,SP,25.78,credit_card,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,13226,varzea paulista,SP,garden_tools,87900,loanda,PR,218.04,credit_card,5.0
